In [1]:
import pandas as pd
import numpy as np
from urllib.request import urlopen
import certifi
import json
import os
import ssl
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Environment variables
import dotenv
dotenv.load_dotenv()
FMP_API_KEY = os.getenv('FMP_API_KEY')
EODHD_API_KEY = os.getenv('EODHD_API_KEY')

## Verify API Key and Test Connection

In [2]:
# Verify API key is loaded
print(f"EODHD API Key loaded: {bool(EODHD_API_KEY)}")
if EODHD_API_KEY:
    print(f"API Key (first 10 chars): {EODHD_API_KEY[:10]}...")
else:
    print("WARNING: EODHD_API_KEY is not set!")
    print("Please ensure you have EODHD_API_KEY in your .env file")

# Test connection with a simple query
print("\nTesting EODHD API connection...")
test_url = f'https://eodhd.com/api/eod/AAPL.US?api_token={EODHD_API_KEY}&fmt=json&limit=5'
try:
    context = ssl.create_default_context(cafile=certifi.where())
    response = urlopen(test_url, context=context)
    data = response.read().decode('utf-8')
    test_data = json.loads(data)
    if test_data and len(test_data) > 0:
        print("✓ API connection successful!")
        print(f"  Retrieved {len(test_data)} test records")
    else:
        print("✗ API returned empty data")
except Exception as e:
    print(f"✗ API test failed: {str(e)}")
    print("\nPossible issues:")
    print("  1. Invalid or expired API key")
    print("  2. Account subscription doesn't include this endpoint")
    print("  3. Network/firewall blocking the request")

EODHD API Key loaded: True
API Key (first 10 chars): 6911297628...

Testing EODHD API connection...
✓ API connection successful!
  Retrieved 11319 test records
✓ API connection successful!
  Retrieved 11319 test records


## Load Existing ETF Symbols

First, let's load the symbols from our existing US equity data to know which ETFs to fetch fundamentals for.

In [3]:
# Load the existing US equity data to get ETF symbols
all_etf_data = pd.read_csv('data/processed/all_etf_data.csv', index_col=0, header=[0, 1], parse_dates=True)

# Extract unique symbols (level 0 of MultiIndex columns)
etf_symbols = all_etf_data.columns.get_level_values(0).unique().tolist()

# Filter out index symbols (those starting with ^)
etf_symbols_only = [symbol for symbol in etf_symbols if not symbol.startswith('^')]

print(f"Total symbols found: {len(etf_symbols)}")
print(f"ETF symbols (excluding indices): {len(etf_symbols_only)}")
print(f"\nETF Symbols: {etf_symbols_only}")

Total symbols found: 47
ETF symbols (excluding indices): 40

ETF Symbols: ['BIL', 'VXUS', 'SHY', 'IEF', 'VEA', 'VWO', 'VGK', 'VPL', 'FXI', 'EWJ', 'INDA', 'SPY', 'VOO', 'RSP', 'ONEQ', 'QQQ', 'IWM', 'DIA', 'IWB', 'IWV', 'XLE', 'XLF', 'XLU', 'XLI', 'XLV', 'XLK', 'XLB', 'XLY', 'XLP', 'XLRE', 'XLC', 'IVW', 'IVE', 'IWO', 'IWN', 'IWF', 'IWD', 'VTHR', 'IWR', 'IWC']


## EODHD API Functions for ETF Fundamentals

EODHD provides comprehensive fundamental data for ETFs including:
- Assets Under Management (AUM)
- Net Asset Value (NAV)
- Expense ratios
- Dividend information
- Holdings data
- Performance metrics

In [4]:
def fetch_etf_fundamentals(symbol, api_key=EODHD_API_KEY):
    """
    Fetch ETF fundamental data from EODHD API
    
    Parameters:
    -----------
    symbol : str
        ETF ticker symbol (e.g., 'SPY')
    api_key : str
        EODHD API key
    
    Returns:
    --------
    dict : ETF fundamental data including general info, technicals, and holdings
    """
    try:
        # EODHD fundamentals endpoint for US ETFs
        # Note: Fundamentals data may require specific subscription level
        url = f'https://eodhd.com/api/fundamentals/{symbol}.US?api_token={api_key}'
        
        context = ssl.create_default_context(cafile=certifi.where())
        response = urlopen(url, context=context)
        data = response.read().decode('utf-8')
        fundamentals = json.loads(data)
        
        if 'General' in fundamentals:
            print(f'✓ {symbol}: Retrieved fundamental data')
            return fundamentals
        else:
            print(f'✗ {symbol}: No fundamental data available')
            return None
            
    except Exception as e:
        error_msg = str(e)
        if '403' in error_msg or 'Forbidden' in error_msg:
            print(f'✗ {symbol}: Access forbidden (403) - Check API subscription level')
        elif '401' in error_msg or 'Unauthorized' in error_msg:
            print(f'✗ {symbol}: Unauthorized (401) - Check API key')
        else:
            print(f'✗ {symbol}: Error - {error_msg}')
        return None

def fetch_etf_historical_nav(symbol, date_from='1990-01-01', date_to='2025-10-24', api_key=EODHD_API_KEY):
    """
    Fetch historical NAV (Net Asset Value) data for an ETF from EODHD
    
    Parameters:
    -----------
    symbol : str
        ETF ticker symbol
    date_from : str
        Start date in YYYY-MM-DD format
    date_to : str
        End date in YYYY-MM-DD format
    api_key : str
        EODHD API key
    
    Returns:
    --------
    pd.DataFrame : Historical NAV data with date index
    """
    try:
        # Historical prices endpoint (includes NAV for ETFs)
        url = f'https://eodhd.com/api/eod/{symbol}.US?from={date_from}&to={date_to}&api_token={api_key}&fmt=json'
        
        context = ssl.create_default_context(cafile=certifi.where())
        response = urlopen(url, context=context)
        data = response.read().decode('utf-8')
        price_data = json.loads(data)
        
        if price_data and len(price_data) > 0:
            df = pd.DataFrame(price_data)
            df['date'] = pd.to_datetime(df['date'])
            df = df.set_index('date').sort_index()
            print(f'✓ {symbol}: {len(df)} observations from {df.index.min().date()} to {df.index.max().date()}')
            return df
        else:
            print(f'✗ {symbol}: No historical data available')
            return None
            
    except Exception as e:
        error_msg = str(e)
        if '403' in error_msg or 'Forbidden' in error_msg:
            print(f'✗ {symbol}: Access forbidden (403) - Check API subscription level')
        elif '401' in error_msg or 'Unauthorized' in error_msg:
            print(f'✗ {symbol}: Unauthorized (401) - Check API key')
        else:
            print(f'✗ {symbol}: Error - {error_msg}')
        return None

def extract_etf_metrics(fundamentals_dict):
    """
    Extract key metrics from ETF fundamentals data
    
    Parameters:
    -----------
    fundamentals_dict : dict
        Dictionary returned from fetch_etf_fundamentals
    
    Returns:
    --------
    dict : Simplified metrics dictionary
    """
    if not fundamentals_dict:
        return None
    
    metrics = {}
    
    # General Information
    if 'General' in fundamentals_dict:
        general = fundamentals_dict['General']
        metrics['symbol'] = general.get('Code', None)
        metrics['name'] = general.get('Name', None)
        metrics['exchange'] = general.get('Exchange', None)
        metrics['currency'] = general.get('CurrencyCode', None)
        metrics['isin'] = general.get('ISIN', None)
        metrics['category'] = general.get('Category', None)
    
    # ETF-specific data
    if 'ETF_Data' in fundamentals_dict:
        etf_data = fundamentals_dict['ETF_Data']
        
        # AUM and NAV
        metrics['net_assets'] = etf_data.get('Net_Assets', None)
        metrics['nav'] = etf_data.get('NAV', None)
        
        # Expense ratios
        metrics['expense_ratio'] = etf_data.get('Expense_Ratio', None)
        
        # Company info
        if 'Company' in etf_data:
            metrics['fund_family'] = etf_data['Company'].get('Name', None)
        
        # Performance metrics
        if 'Technicals' in etf_data:
            tech = etf_data['Technicals']
            metrics['beta'] = tech.get('Beta', None)
            metrics['52w_high'] = tech.get('52WeekHigh', None)
            metrics['52w_low'] = tech.get('52WeekLow', None)
            metrics['50d_ma'] = tech.get('50DayMA', None)
            metrics['200d_ma'] = tech.get('200DayMA', None)
        
        # Asset allocation
        if 'Asset_Allocation' in etf_data:
            metrics['asset_allocation'] = etf_data['Asset_Allocation']
        
        # Top holdings
        if 'Holdings' in etf_data:
            metrics['top_holdings'] = etf_data['Holdings']
    
    # Valuation metrics
    if 'Technicals' in fundamentals_dict:
        tech = fundamentals_dict['Technicals']
        metrics['beta_general'] = tech.get('Beta', None)
    
    return metrics

## Fetch ETF Fundamentals Data

Now let's retrieve fundamental data for all ETFs in our dataset.

In [5]:
# Fetch current fundamental data for all ETFs
print(f"Fetching fundamental data for {len(etf_symbols_only)} ETFs...")
print("=" * 80)

etf_fundamentals = {}
etf_metrics_list = []
errors_count = {'403': 0, '401': 0, 'other': 0}

for i, symbol in enumerate(etf_symbols_only, 1):
    print(f"[{i}/{len(etf_symbols_only)}] ", end='')
    
    # Fetch raw fundamentals
    fund_data = fetch_etf_fundamentals(symbol)
    if fund_data:
        etf_fundamentals[symbol] = fund_data
        
        # Extract key metrics
        metrics = extract_etf_metrics(fund_data)
        if metrics:
            etf_metrics_list.append(metrics)
    else:
        # Track error types for debugging
        # This will be updated in the function above
        pass

# Create summary DataFrame
etf_metrics_df = pd.DataFrame(etf_metrics_list)

print("\n" + "=" * 80)
print(f"Successfully retrieved fundamentals for {len(etf_fundamentals)} / {len(etf_symbols_only)} ETFs")

if len(etf_fundamentals) == 0:
    print("\n⚠️  WARNING: No fundamental data retrieved!")
    print("\nTroubleshooting steps:")
    print("1. Verify your EODHD API key is correct")
    print("2. Check if your subscription includes 'Fundamental Data' API access")
    print("3. Visit: https://eodhd.com/cp/settings to check your subscription")
    print("4. Consider using alternative data source (e.g., Yahoo Finance via yfinance)")
else:
    print(f"\nSummary DataFrame shape: {etf_metrics_df.shape}")
    print("\nAvailable columns:")
    print(etf_metrics_df.columns.tolist())

Fetching fundamental data for 40 ETFs...
[1/40] ✓ BIL: Retrieved fundamental data
[2/40] ✓ BIL: Retrieved fundamental data
[2/40] ✓ VXUS: Retrieved fundamental data
[3/40] ✓ SHY: Retrieved fundamental data
[4/40] ✓ VXUS: Retrieved fundamental data
[3/40] ✓ SHY: Retrieved fundamental data
[4/40] ✓ IEF: Retrieved fundamental data
[5/40] ✓ IEF: Retrieved fundamental data
[5/40] ✓ VEA: Retrieved fundamental data
[6/40] ✓ VEA: Retrieved fundamental data
[6/40] ✓ VWO: Retrieved fundamental data
[7/40] ✓ VWO: Retrieved fundamental data
[7/40] ✓ VGK: Retrieved fundamental data
[8/40] ✓ VGK: Retrieved fundamental data
[8/40] ✓ VPL: Retrieved fundamental data
[9/40] ✓ FXI: Retrieved fundamental data
[10/40] ✓ VPL: Retrieved fundamental data
[9/40] ✓ FXI: Retrieved fundamental data
[10/40] ✓ EWJ: Retrieved fundamental data
[11/40] ✓ EWJ: Retrieved fundamental data
[11/40] ✓ INDA: Retrieved fundamental data
[12/40] ✓ SPY: Retrieved fundamental data
[13/40] ✓ INDA: Retrieved fundamental data
[12/40

## Display Sample Fundamentals Data

In [8]:
# Display sample of the metrics
print("Sample ETF Metrics:")
print("=" * 80)
display(etf_metrics_df.head(10))

# Show key statistics
print("\n\nKey Statistics:")
print("=" * 80)
print(f"Average Expense Ratio: {etf_metrics_df['expense_ratio'].mean():.4f}%")
print(f"Average Beta: {etf_metrics_df['beta_general'].mean():.4f}")
print(f"\nFund Families:")

Sample ETF Metrics:


,symbol,name,exchange,currency,isin,category,net_assets,nav,expense_ratio,asset_allocation,top_holdings,beta_general
0,BIL,SPDR® Bloomberg 1-3 Month T-Bill ETF,NYSE ARCA,USD,None,Ultrashort Bond,None,None,None,"{'Cash': {'Long_%': '100', 'Short_%': '0', 'Ne...",{'United States Treasury Bills 0.01%': {'Name'...,0.06
1,VXUS,Vanguard Total International Stock Index Fund ...,NASDAQ,USD,None,Foreign Large Blend,None,None,None,"{'Cash': {'Long_%': '2.22738', 'Short_%': '0.0...","{'2330.TW': {'Code': '2330', 'Exchange': 'TW',...",1.04
2,SHY,iShares 1-3 Year Treasury Bond ETF,NASDAQ,USD,None,Short Government,None,None,None,"{'Cash': {'Long_%': '0.37844', 'Short_%': '0.3...",{'United States Treasury Notes 0.38%': {'Name'...,0.24
3,IEF,iShares 7-10 Year Treasury Bond ETF,NASDAQ,USD,None,Long Government,None,None,None,"{'Cash': {'Long_%': '0.34265', 'Short_%': '0.3...",{'United States Treasury Notes 0.88%': {'Name'...,1.16
4,VEA,Vanguard FTSE Developed Markets Index Fund ETF...,NYSE ARCA,USD,None,Foreign Large Blend,None,None,None,"{'Cash': {'Long_%': '1.19196', 'Short_%': '0.2...","{'ASML.AS': {'Code': 'ASML', 'Exchange': 'AS',...",1.08
5,VWO,Vanguard FTSE Emerging Markets Index Fund ETF ...,NYSE ARCA,USD,None,Diversified Emerging Mkts,None,None,None,"{'Cash': {'Long_%': '3.8298', 'Short_%': '0.13...","{'2330.TW': {'Code': '2330', 'Exchange': 'TW',...",0.94
6,VGK,Vanguard FTSE Europe Index Fund ETF Shares,NYSE ARCA,USD,None,Europe Stock,None,None,None,"{'Cash': {'Long_%': '0.91667', 'Short_%': '0.0...","{'ASML.AS': {'Code': 'ASML', 'Exchange': 'AS',...",1.12
7,VPL,Vanguard FTSE Pacific Index Fund ETF Shares,NYSE ARCA,USD,None,Diversified Pacific/Asia,None,None,None,"{'Cash': {'Long_%': '1.48628', 'Short_%': '0.0...","{'SSNLF.US': {'Code': 'SSNLF', 'Exchange': 'US...",1.05
8,FXI,iShares China Large-Cap ETF,NYSE ARCA,USD,None,China Region,None,None,None,"{'Cash': {'Long_%': '0.3021', 'Short_%': '0.21...","{'9988.HK': {'Code': '9988', 'Exchange': 'HK',...",1.39
9,EWJ,iShares MSCI Japan ETF,NYSE ARCA,USD,None,Japan Stock,None,None,None,"{'Cash': {'Long_%': '1.08204', 'Short_%': '1.0...","{'7203.TSE': {'Code': '7203', 'Exchange': 'TSE...",0.88




Key Statistics:
Average Expense Ratio: nan%
Average Beta: 0.9957

Fund Families:


## Fetch Historical NAV Data

Retrieve historical Net Asset Value (NAV) data for all ETFs from the earliest available date to 2025-10-24.

In [9]:
# Fetch historical NAV/price data for all ETFs
print(f"Fetching historical NAV data for {len(etf_symbols_only)} ETFs...")
print("=" * 80)

date_from = '1990-01-01'
date_to = '2025-10-24'

etf_nav_data = {}

for symbol in etf_symbols_only:
    nav_df = fetch_etf_historical_nav(symbol, date_from=date_from, date_to=date_to)
    if nav_df is not None:
        # Add symbol as column prefix for MultiIndex structure
        nav_df.columns = pd.MultiIndex.from_product([[symbol], nav_df.columns])
        etf_nav_data[symbol] = nav_df

print("\n" + "=" * 80)
print(f"Successfully retrieved historical data for {len(etf_nav_data)} / {len(etf_symbols_only)} ETFs")

Fetching historical NAV data for 40 ETFs...
✓ BIL: 4633 observations from 2007-05-30 to 2025-10-24
✓ BIL: 4633 observations from 2007-05-30 to 2025-10-24
✓ VXUS: 3708 observations from 2011-01-28 to 2025-10-24
✓ VXUS: 3708 observations from 2011-01-28 to 2025-10-24
✓ SHY: 5851 observations from 2002-07-26 to 2025-10-24
✓ SHY: 5851 observations from 2002-07-26 to 2025-10-24
✓ IEF: 5851 observations from 2002-07-26 to 2025-10-24
✓ IEF: 5851 observations from 2002-07-26 to 2025-10-24
✓ VEA: 4593 observations from 2007-07-26 to 2025-10-24
✓ VEA: 4593 observations from 2007-07-26 to 2025-10-24
✓ VWO: 5191 observations from 2005-03-10 to 2025-10-24
✓ VWO: 5191 observations from 2005-03-10 to 2025-10-24
✓ VGK: 5191 observations from 2005-03-10 to 2025-10-24
✓ VGK: 5191 observations from 2005-03-10 to 2025-10-24
✓ VPL: 5191 observations from 2005-03-10 to 2025-10-24
✓ VPL: 5191 observations from 2005-03-10 to 2025-10-24
✓ FXI: 5296 observations from 2004-10-08 to 2025-10-24
✓ FXI: 5296 observa

## Combine and Save ETF Data

In [10]:
# Combine all historical NAV data into single DataFrame
if etf_nav_data:
    all_etf_nav = pd.concat(list(etf_nav_data.values()), axis=1)
    
    print(f"\nCombined NAV DataFrame:")
    print(f"  Shape: {all_etf_nav.shape}")
    print(f"  Date range: {all_etf_nav.index.min()} to {all_etf_nav.index.max()}")
    print(f"  Number of ETFs: {len(etf_nav_data)}")
    
    # Save historical NAV data
    all_etf_nav.to_csv('data/processed/etf_nav_data.csv')
    print(f"\n✓ Saved historical NAV data to: data/processed/etf_nav_data.csv")
else:
    print("No NAV data retrieved")

# Save current fundamentals metrics
etf_metrics_df.to_csv('data/processed/etf_fundamentals_metrics.csv', index=False)
print(f"✓ Saved fundamental metrics to: data/processed/etf_fundamentals_metrics.csv")

# Save raw fundamentals as JSON for detailed analysis
import json
with open('data/processed/etf_fundamentals_raw.json', 'w') as f:
    json.dump(etf_fundamentals, f, indent=2)
print(f"✓ Saved raw fundamental data to: data/processed/etf_fundamentals_raw.json")

print("\n" + "=" * 80)
print("DATA RETRIEVAL COMPLETE")
print("=" * 80)


Combined NAV DataFrame:
  Shape: (8242, 240)
  Date range: 1993-01-29 00:00:00 to 2025-10-24 00:00:00
  Number of ETFs: 40

✓ Saved historical NAV data to: data/processed/etf_nav_data.csv
✓ Saved fundamental metrics to: data/processed/etf_fundamentals_metrics.csv

✓ Saved historical NAV data to: data/processed/etf_nav_data.csv
✓ Saved fundamental metrics to: data/processed/etf_fundamentals_metrics.csv
✓ Saved raw fundamental data to: data/processed/etf_fundamentals_raw.json

DATA RETRIEVAL COMPLETE
✓ Saved raw fundamental data to: data/processed/etf_fundamentals_raw.json

DATA RETRIEVAL COMPLETE


## Summary Statistics and Verification

In [11]:
# Summary of data coverage
print("ETF FUNDAMENTAL DATA SUMMARY")
print("=" * 80)

print(f"\n1. Current Fundamentals Metrics:")
print(f"   - ETFs with data: {len(etf_metrics_df)}")
print(f"   - Metrics captured: {len(etf_metrics_df.columns)}")

print(f"\n2. Historical NAV Data:")
if etf_nav_data:
    print(f"   - ETFs with historical data: {len(etf_nav_data)}")
    print(f"   - Total observations: {all_etf_nav.shape[0]:,}")
    print(f"   - Date range: {all_etf_nav.index.min().date()} to {all_etf_nav.index.max().date()}")
    
    # Check data availability by ETF
    print(f"\n3. Data Availability by ETF:")
    nav_counts = []
    for symbol in etf_nav_data.keys():
        count = all_etf_nav[symbol].notna().sum(axis=1).sum()
        start_date = all_etf_nav[symbol].first_valid_index()
        nav_counts.append({
            'symbol': symbol,
            'observations': count,
            'start_date': start_date
        })
    
    nav_summary = pd.DataFrame(nav_counts).sort_values('start_date')
    print(f"\n   Earliest data availability:")
    display(nav_summary.head(10))
    
    print(f"\n   Most recent additions:")
    display(nav_summary.tail(5))

print("\n" + "=" * 80)

ETF FUNDAMENTAL DATA SUMMARY

1. Current Fundamentals Metrics:
   - ETFs with data: 40
   - Metrics captured: 12

2. Historical NAV Data:
   - ETFs with historical data: 40
   - Total observations: 8,242
   - Date range: 1993-01-29 to 2025-10-24

3. Data Availability by ETF:

   Earliest data availability:


,symbol,observations,start_date
11,SPY,49452,1993-01-29
9,EWJ,44706,1996-03-18
17,DIA,41916,1998-01-20
28,XLP,40512,1998-12-22
27,XLY,40512,1998-12-22
26,XLB,40512,1998-12-22
25,XLK,40512,1998-12-22
24,XLV,40512,1998-12-22
23,XLI,40506,1998-12-22
22,XLU,40512,1998-12-22



   Most recent additions:


,symbol,observations,start_date
37,VTHR,22434,2010-09-22
1,VXUS,22248,2011-01-28
10,INDA,20562,2012-02-03
29,XLRE,15162,2015-10-08
30,XLC,11094,2018-06-19
